# Particle identification with deep learning

This notebook revisits the particle-identification problem from the first lecture using neural networks. We will train fully connected networks on reconstructed track features, then build a convolutional network that learns directly from two-dimensional detector projections.

The emphasis is on a reliable workflow: separate training, validation, and test samples; fit preprocessing on the training sample only; monitor generalization during training; and evaluate the final model on untouched test events.


## Prerequisites


The first cell verifies the public teaching dataset and plotting helpers, downloading a clean copy when either file is missing or incomplete. Downloads are pinned to the same tested repository revision as the first lecture.

TensorFlow will use a GPU when one is available, but every section also runs on a CPU. In Google Colab, a GPU can be selected under **Runtime → Change runtime type → Hardware accelerator**.


In [ ]:
from pathlib import Path
import hashlib
import os
import pickle
import shutil
import time
from urllib.request import Request, urlopen

resources = {
    "modules.py": {
        "url": "https://raw.githubusercontent.com/saulam/neutrinoml/10e610a3bf7a628c38fd3ece00e67e195e0b5ca6/modules.py",
        "size": 10041,
        "sha256": "faebdba3b213e6ebec48f99d573cb582ccc08f008ce996dbc852b141e262d190",
    },
    "df_pgun_teaching.p": {
        "url": "https://raw.githubusercontent.com/saulam/neutrinoml/10e610a3bf7a628c38fd3ece00e67e195e0b5ca6/df_pgun_teaching.p",
        "size": 79089990,
        "sha256": "98d4f7fced6a1c9c501d8e468dd20ddc1cec23878d2dc8323a95f4ff20eb98d4",
    },
}


def file_sha256(path):
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def ensure_resource(filename, metadata, attempts=3):
    path = Path(filename)
    if (
        path.is_file()
        and path.stat().st_size == metadata["size"]
        and file_sha256(path) == metadata["sha256"]
    ):
        return

    temporary_path = path.with_name(f".{path.name}.download")
    last_error = None

    for attempt in range(attempts):
        try:
            request = Request(
                metadata["url"],
                headers={"User-Agent": "neutrino-ml-teaching"},
            )
            with urlopen(request, timeout=120) as response, temporary_path.open("wb") as output:
                shutil.copyfileobj(response, output, length=1024 * 1024)

            if temporary_path.stat().st_size != metadata["size"]:
                raise OSError("downloaded file has the wrong size")
            if file_sha256(temporary_path) != metadata["sha256"]:
                raise OSError("downloaded file failed its checksum")

            temporary_path.replace(path)
            return
        except Exception as error:
            last_error = error
            temporary_path.unlink(missing_ok=True)
            if attempt + 1 < attempts:
                time.sleep(2**attempt)

    raise RuntimeError(f"Could not obtain a valid copy of {filename}") from last_error


for filename, metadata in resources.items():
    ensure_resource(filename, metadata)

os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf

from IPython.display import display
from sklearn.metrics import ConfusionMatrixDisplay, classification_report
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from tensorflow import keras
from tensorflow.keras import layers

from modules import plot_event, plot_params_pid

RANDOM_STATE = 7
PID_TO_LABEL = {2212: 0, 13: 1, 211: 2}
CLASS_NAMES = ["proton", "muon", "pion"]

gpus = tf.config.list_physical_devices("GPU")
for gpu in gpus:
    try:
        tf.config.experimental.set_memory_growth(gpu, True)
    except RuntimeError:
        # Memory growth must be set before a GPU has been initialized.
        pass

tf.keras.utils.set_random_seed(RANDOM_STATE)

accelerator = gpus[0].name if gpus else "CPU"
print(f"TensorFlow {tf.__version__}; accelerator: {accelerator}")


## Dataset


Load the event-level DataFrame from the downloaded pickle file.


In [ ]:
with Path("df_pgun_teaching.p").open("rb") as stream:
    df = pickle.load(stream, encoding="latin1")

df = df.reset_index(drop=True)


The dataset contains 59,578 simulated particle-gun events. Its columns are:

- **TruePID**: PDG particle identifier: 2212 (proton), 13 (muon), or 211 (charged pion).
- **TrueMomentum**: generated particle momentum in MeV/$c$.
- **NNodes**: number of reconstructed nodes in the track.
- **NodeOrder**: ordering of the nodes along the reconstructed track.
- **NodePosX**, **NodePosY**, **NodePosZ**: node coordinates in mm.
- **NodeT**: node timestamps in ns.
- **Nodededx**: node-level energy-loss proxy.
- **TrkLen**: reconstructed track length in mm.
- **TrkEDepo**: total track energy-deposition proxy.
- **TrkDir1**, **TrkDir2**: reconstructed track-direction variables.


In [ ]:
df.head()


The three particle species are represented in comparable, although not identical, numbers.


In [ ]:
class_counts = (
    df["TruePID"]
    .map(PID_TO_LABEL)
    .value_counts()
    .sort_index()
    .rename(index=dict(enumerate(CLASS_NAMES)))
    .rename("events")
)
class_counts


Reconstructed tracks contain a variable number of three-dimensional nodes. Change `event_number` to inspect another event.


In [ ]:
event_number = 5
plot_event(df, event_number)


## Common utilities


Neural-network training uses three disjoint samples:

- the **training sample** updates the network weights;
- the **validation sample** selects the stopping point and diagnoses overfitting;
- the **test sample** is used only for the final evaluation.

To stay comparable with the first lecture, 40% of the events are reserved for testing. Twenty percent of the remaining training pool is used for validation, giving an overall 48%/12%/40% split. Every split is stratified by particle type.


In [ ]:
FEATURE_NAMES = [
    "dE/dx first node 1",
    "dE/dx first node 2",
    "dE/dx first node 3",
    "dE/dx node n-4",
    "dE/dx node n-3",
    "dE/dx node n-2",
    "dE/dx node n-1",
    "dE/dx node n",
    "TrkLen",
    "TrkEDepo",
    "TrkDir1",
    "TrkDir2",
]


def split_train_validation_test(features, labels):
    X_train_pool, X_test, y_train_pool, y_test = train_test_split(
        features,
        labels,
        test_size=0.4,
        random_state=RANDOM_STATE,
        stratify=labels,
    )
    X_train, X_validation, y_train, y_validation = train_test_split(
        X_train_pool,
        y_train_pool,
        test_size=0.2,
        random_state=RANDOM_STATE,
        stratify=y_train_pool,
    )
    return X_train, X_validation, X_test, y_train, y_validation, y_test


def build_track_features(frame):
    # Keep eight ordered dE/dx values and four global track features.
    features = np.full((len(frame), 12), -1.0, dtype=np.float32)

    for row_number, event in enumerate(frame.itertuples(index=False)):
        order = np.asarray(event.NodeOrder, dtype=int)
        dedx = np.asarray(event.Nodededx, dtype=np.float32)[order]

        first = dedx[:3]
        tail = dedx[len(first):][-5:]
        features[row_number, :len(first)] = first
        features[row_number, 8 - len(tail):8] = tail
        features[row_number, 8:] = (
            event.TrkLen,
            event.TrkEDepo,
            event.TrkDir1,
            event.TrkDir2,
        )

    return features


def plot_feature_distributions(features, labels, feature_names, class_names):
    fig, axes = plt.subplots(4, 3, figsize=(15, 12))
    for feature_number, axis in enumerate(axes.flat):
        for class_number, class_name in enumerate(class_names):
            values = features[labels == class_number, feature_number]
            if feature_number < 8:
                values = values[values != -1]
            axis.hist(
                values,
                bins=50,
                histtype="step",
                density=True,
                label=class_name,
            )
        axis.set_xlabel(feature_names[feature_number])
        axis.set_ylabel("density")
        axis.legend()
    fig.tight_layout()
    plt.show()


The following helpers define, train, and evaluate the models. Early stopping restores the weights from the epoch with the lowest validation loss.


In [ ]:
MAX_EPOCHS = 30
BATCH_SIZE = 128


def make_dense_classifier(n_features, n_classes, hidden_units=(10, 10), dropout=0.0):
    inputs = keras.Input(shape=(n_features,), name="features")
    x = inputs
    for layer_number, units in enumerate(hidden_units, start=1):
        x = layers.Dense(units, activation="relu", name=f"hidden_{layer_number}")(x)
        if dropout:
            x = layers.Dropout(dropout, name=f"dropout_{layer_number}")(x)

    if n_classes == 2:
        outputs = layers.Dense(1, activation="sigmoid", name="class_probability")(x)
        loss = "binary_crossentropy"
    else:
        outputs = layers.Dense(n_classes, activation="softmax", name="class_probabilities")(x)
        loss = "sparse_categorical_crossentropy"

    model = keras.Model(inputs, outputs)
    model.compile(optimizer=keras.optimizers.Adam(), loss=loss, metrics=["accuracy"])
    return model


def early_stopping():
    return keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=4,
        restore_best_weights=True,
    )


def plot_history(history):
    history_frame = pd.DataFrame(history.history)
    epochs = np.arange(1, len(history_frame) + 1)

    fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
    axes[0].plot(epochs, history_frame["loss"], label="training")
    axes[0].plot(epochs, history_frame["val_loss"], label="validation")
    axes[0].set(xlabel="epoch", ylabel="loss")
    axes[0].legend()

    axes[1].plot(epochs, history_frame["accuracy"], label="training")
    axes[1].plot(epochs, history_frame["val_accuracy"], label="validation")
    axes[1].set(xlabel="epoch", ylabel="accuracy", ylim=(0, 1))
    axes[1].legend()

    fig.tight_layout()
    plt.show()


def evaluation_table(y_true, y_pred, class_names):
    report = classification_report(
        y_true,
        y_pred,
        labels=np.arange(len(class_names)),
        target_names=class_names,
        output_dict=True,
        zero_division=0,
    )
    columns = ["precision", "recall", "f1-score", "support"]
    rows = [*class_names, "macro avg", "weighted avg"]
    return pd.DataFrame(report).T.loc[rows, columns].round(3)


def show_evaluation(y_true, y_pred, class_names):
    display(evaluation_table(y_true, y_pred, class_names))
    ConfusionMatrixDisplay.from_predictions(
        y_true,
        y_pred,
        labels=np.arange(len(class_names)),
        display_labels=class_names,
        cmap="Blues",
        colorbar=False,
        values_format="d",
    )
    plt.show()


## Fully connected network: two features


Begin with the binary proton–muon task and the same two global features used for logistic regression in the first lecture: track length and total energy deposition. Pions are excluded for now.

The scaler is fitted on the training sample only. Applying a scaler fitted to all events would leak information from the validation and test samples into training.


In [ ]:
binary_df = df[df["TruePID"].isin([2212, 13])].reset_index(drop=True)
X2 = binary_df[["TrkLen", "TrkEDepo"]].to_numpy(dtype=np.float32)
y2 = binary_df["TruePID"].map({2212: 0, 13: 1}).to_numpy(dtype=np.int32)

(
    X2_train_raw,
    X2_validation_raw,
    X2_test_raw,
    y2_train,
    y2_validation,
    y2_test,
) = split_train_validation_test(X2, y2)

scaler2 = StandardScaler()
X2_train = scaler2.fit_transform(X2_train_raw).astype(np.float32)
X2_validation = scaler2.transform(X2_validation_raw).astype(np.float32)
X2_test = scaler2.transform(X2_test_raw).astype(np.float32)

pd.Series(
    {"training": len(X2_train), "validation": len(X2_validation), "test": len(X2_test)},
    name="events",
)


Inspect the training sample. The class distributions are distinct but overlap, so perfect separation is not expected.


In [ ]:
plot_params_pid(
    X2_train_raw,
    y2_train,
    param_names=["track length [mm]", "deposited-energy proxy"],
    y_names=["proton", "muon"],
)


A dense layer computes a learned affine transformation followed by a nonlinear activation. For one neuron,

$$
h = \sigma(\mathbf{x}^{\mathsf{T}}\mathbf{w} + b).
$$

Rectified linear units (ReLUs) provide the hidden-layer nonlinearities. The output neuron uses a sigmoid, so its output is interpreted as $P(y=1\mid\mathbf{x})$, the estimated muon probability.

Binary cross-entropy is minimized during training:

$$
\mathcal{L}_{\mathrm{BCE}}
= -\frac{1}{m}\sum_{i=1}^{m}
\left[y_i\log(\hat y_i) + (1-y_i)\log(1-\hat y_i)\right].
$$

<div>
<img src="https://raw.githubusercontent.com/saulam/neutrinoml/10e610a3bf7a628c38fd3ece00e67e195e0b5ca6/dense_nn.png" width="700"/>
</div>


In [ ]:
tf.keras.utils.set_random_seed(RANDOM_STATE)
dense2_model = make_dense_classifier(
    n_features=2,
    n_classes=2,
    hidden_units=(10, 10),
)
dense2_model.summary()


The network trains on mini-batches of 128 events. One **epoch** is one pass over the training sample; one **batch** produces one gradient update. Training can run for at most 30 epochs, but early stopping usually ends it sooner when validation loss no longer improves.


In [ ]:
dense2_history = dense2_model.fit(
    X2_train,
    y2_train,
    validation_data=(X2_validation, y2_validation),
    epochs=MAX_EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=[early_stopping()],
    verbose=1,
)


Compare training and validation curves. A widening gap—training loss falling while validation loss rises—is a sign of overfitting.


In [ ]:
plot_history(dense2_history)


Evaluate the selected model once on the held-out test events. A sigmoid output of at least 0.5 is assigned to the muon class. In the confusion matrix, rows are true classes and columns are predicted classes.


In [ ]:
dense2_probability = dense2_model.predict(X2_test, verbose=0).ravel()
dense2_prediction = (dense2_probability >= 0.5).astype(np.int32)

show_evaluation(y2_test, dense2_prediction, ["proton", "muon"])


With only two inputs, the neural network has little information beyond that available to logistic regression. Its advantage is the ability to learn a nonlinear boundary; it is not automatically a better classifier.


## Fully connected network: 12 features


A fixed-size representation can retain the first three ordered node values, the last five remaining node values, and four global track features. Missing node positions are filled with $-1$ and later standardized with the other features.

<div>
<img src="https://raw.githubusercontent.com/saulam/neutrinoml/10e610a3bf7a628c38fd3ece00e67e195e0b5ca6/reg.png" width="500"/>
</div>


In [ ]:
X12_binary = build_track_features(binary_df)
y12_binary = binary_df["TruePID"].map({2212: 0, 13: 1}).to_numpy(dtype=np.int32)

(
    X12_train_raw,
    X12_validation_raw,
    X12_test_raw,
    y12_train,
    y12_validation,
    y12_test,
) = split_train_validation_test(X12_binary, y12_binary)

scaler12 = StandardScaler()
X12_train = scaler12.fit_transform(X12_train_raw).astype(np.float32)
X12_validation = scaler12.transform(X12_validation_raw).astype(np.float32)
X12_test = scaler12.transform(X12_test_raw).astype(np.float32)

pd.DataFrame(X12_train_raw, columns=FEATURE_NAMES).head()


Class-conditional histograms of the training events reveal which inputs may be discriminative. The padded values are omitted from the first eight histograms.


In [ ]:
plot_feature_distributions(
    X12_train_raw,
    y12_train,
    feature_names=FEATURE_NAMES,
    class_names=["proton", "muon"],
)


Increase the hidden-layer width modestly to accommodate the larger input. The evaluation procedure and untouched test sample remain unchanged.


In [ ]:
tf.keras.utils.set_random_seed(RANDOM_STATE)
dense12_model = make_dense_classifier(
    n_features=12,
    n_classes=2,
    hidden_units=(32, 16),
)

dense12_history = dense12_model.fit(
    X12_train,
    y12_train,
    validation_data=(X12_validation, y12_validation),
    epochs=MAX_EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=[early_stopping()],
    verbose=1,
)


In [ ]:
plot_history(dense12_history)

dense12_probability = dense12_model.predict(X12_test, verbose=0).ravel()
dense12_prediction = (dense12_probability >= 0.5).astype(np.int32)
show_evaluation(y12_test, dense12_prediction, ["proton", "muon"])


The node-level inputs retain information that is absent from the two global features. The class-specific precision and recall show whether the improvement is shared by both particle types.


## Multiclass dense network


Include charged pions by changing the output to three units. A softmax converts the three logits $z_k$ into normalized class probabilities,

$$
P(y=k\mid\mathbf{x}) = \frac{e^{z_k}}{\sum_j e^{z_j}}.
$$

Sparse categorical cross-entropy accepts the integer labels 0, 1, and 2 directly. The most probable class is used as the prediction.


In [ ]:
X12_multiclass = build_track_features(df)
y12_multiclass = df["TruePID"].map(PID_TO_LABEL).to_numpy(dtype=np.int32)

(
    X3_train_raw,
    X3_validation_raw,
    X3_test_raw,
    y3_train,
    y3_validation,
    y3_test,
) = split_train_validation_test(X12_multiclass, y12_multiclass)

scaler3 = StandardScaler()
X3_train = scaler3.fit_transform(X3_train_raw).astype(np.float32)
X3_validation = scaler3.transform(X3_validation_raw).astype(np.float32)
X3_test = scaler3.transform(X3_test_raw).astype(np.float32)


In [ ]:
tf.keras.utils.set_random_seed(RANDOM_STATE)
dense3_model = make_dense_classifier(
    n_features=12,
    n_classes=3,
    hidden_units=(64, 32),
    dropout=0.1,
)
dense3_model.summary()

dense3_history = dense3_model.fit(
    X3_train,
    y3_train,
    validation_data=(X3_validation, y3_validation),
    epochs=MAX_EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=[early_stopping()],
    verbose=1,
)


In [ ]:
plot_history(dense3_history)

dense3_probability = dense3_model.predict(X3_test, verbose=0)
dense3_prediction = dense3_probability.argmax(axis=1)
show_evaluation(y3_test, dense3_prediction, CLASS_NAMES)


The three-class task is more difficult because pion signatures overlap those of protons and muons. Dropout randomly removes a fraction of hidden activations during training, which discourages the network from relying on a small set of neurons.


## Convolutional neural network


A convolutional neural network (CNN) learns local spatial patterns with shared filters. Here each event is projected onto the detector's $YZ$ plane, preserving the beam direction along $Z$. Nodes that fall in the same pixel are averaged. The resulting energy-loss values are divided by a fixed factor of 1,000 to place them in a convenient numerical range without fitting anything to the data.

The projection is produced event by event. This avoids materializing the full image dataset, which would require about 2.6 GB as a single dense `float32` array.


In [ ]:
IMAGE_HEIGHT = 58
IMAGE_WIDTH = 189
IMAGE_SHAPE = (IMAGE_HEIGHT, IMAGE_WIDTH, 1)
Y_MIN = -257.56
Z_MIN = -2888.78
VOXEL_SIZE = 10.0


def event_projection(event):
    y_coordinate = np.asarray(event.NodePosY, dtype=np.float32)
    z_coordinate = np.asarray(event.NodePosZ, dtype=np.float32)
    dedx = np.asarray(event.Nodededx, dtype=np.float32)

    y_pixel = np.floor((y_coordinate - Y_MIN) / VOXEL_SIZE).astype(int)
    z_pixel = np.floor((z_coordinate - Z_MIN) / VOXEL_SIZE).astype(int)
    y_pixel = np.clip(y_pixel, 0, IMAGE_HEIGHT - 1)
    z_pixel = np.clip(z_pixel, 0, IMAGE_WIDTH - 1)

    sums = np.zeros((IMAGE_HEIGHT, IMAGE_WIDTH), dtype=np.float32)
    counts = np.zeros((IMAGE_HEIGHT, IMAGE_WIDTH), dtype=np.int32)
    np.add.at(sums, (y_pixel, z_pixel), dedx)
    np.add.at(counts, (y_pixel, z_pixel), 1)

    image = np.zeros_like(sums)
    np.divide(sums, counts, out=image, where=counts > 0)
    return (image / 1000.0)[..., np.newaxis]


Compare a reconstructed track with its $YZ$ projection.


In [ ]:
event_number = 5
plot_event(df, event_number)

example_projection = event_projection(df.iloc[event_number])
plt.figure(figsize=(10, 3.5))
plt.imshow(
    example_projection[..., 0],
    origin="lower",
    aspect="auto",
    cmap="Reds",
)
plt.xlabel("Z pixel")
plt.ylabel("Y pixel")
plt.title("YZ projection: mean dE/dx divided by 1,000")
plt.colorbar(label="scaled energy-loss proxy")
plt.show()


For a practical lecture runtime, the CNN section uses at most 4,000 events from each class. Set `CNN_EVENTS_PER_CLASS = None` to use every event. Sampling occurs before a new stratified training/validation/test split, so the CNN test events remain untouched during training.


In [ ]:
CNN_EVENTS_PER_CLASS = 4_000
all_labels = df["TruePID"].map(PID_TO_LABEL).to_numpy(dtype=np.int32)


def balanced_event_indices(labels, events_per_class):
    if events_per_class is None:
        return np.arange(len(labels), dtype=int)

    generator = np.random.default_rng(RANDOM_STATE)
    selected = []
    for class_number in range(len(CLASS_NAMES)):
        candidates = np.flatnonzero(labels == class_number)
        sample_size = min(events_per_class, len(candidates))
        selected.append(generator.choice(candidates, size=sample_size, replace=False))
    return generator.permutation(np.concatenate(selected))


cnn_indices = balanced_event_indices(all_labels, CNN_EVENTS_PER_CLASS)
cnn_train_pool, cnn_test_indices = train_test_split(
    cnn_indices,
    test_size=0.4,
    random_state=RANDOM_STATE,
    stratify=all_labels[cnn_indices],
)
cnn_train_indices, cnn_validation_indices = train_test_split(
    cnn_train_pool,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=all_labels[cnn_train_pool],
)

pd.Series(
    {
        "training": len(cnn_train_indices),
        "validation": len(cnn_validation_indices),
        "test": len(cnn_test_indices),
    },
    name="events",
)


A `tf.data` pipeline constructs only the images needed for the current batches. The test pipeline is not shuffled, so its predictions remain aligned with `cnn_test_indices`.


In [ ]:
def make_projection_dataset(indices, training=False):
    indices = np.asarray(indices, dtype=int)

    def generate_events():
        for row_number in indices:
            yield event_projection(df.iloc[row_number]), all_labels[row_number]

    dataset = tf.data.Dataset.from_generator(
        generate_events,
        output_signature=(
            tf.TensorSpec(shape=IMAGE_SHAPE, dtype=tf.float32),
            tf.TensorSpec(shape=(), dtype=tf.int32),
        ),
    )
    if training:
        dataset = dataset.shuffle(
            buffer_size=min(len(indices), 2_048),
            seed=RANDOM_STATE,
            reshuffle_each_iteration=True,
        )
    dataset = dataset.batch(BATCH_SIZE)
    number_of_batches = int(np.ceil(len(indices) / BATCH_SIZE))
    dataset = dataset.apply(
        tf.data.experimental.assert_cardinality(number_of_batches)
    )
    return dataset.prefetch(tf.data.AUTOTUNE)


cnn_train_dataset = make_projection_dataset(cnn_train_indices, training=True)
cnn_validation_dataset = make_projection_dataset(cnn_validation_indices)
cnn_test_dataset = make_projection_dataset(cnn_test_indices)


The first convolution uses a wide kernel to capture extended track segments. Strided convolutions and max pooling reduce the spatial resolution, and the final feature maps are flattened before the dense and softmax layers.

<div>
<img src="https://raw.githubusercontent.com/saulam/neutrinoml/10e610a3bf7a628c38fd3ece00e67e195e0b5ca6/cnn.png" width="900"/>
</div>


In [ ]:
tf.keras.utils.set_random_seed(RANDOM_STATE)

cnn_model = keras.Sequential(
    [
        keras.Input(shape=IMAGE_SHAPE),
        layers.Conv2D(
            16,
            kernel_size=(6, 18),
            strides=(2, 3),
            activation="relu",
        ),
        layers.MaxPooling2D(pool_size=(2, 3), strides=(2, 3)),
        layers.Conv2D(
            32,
            kernel_size=(3, 3),
            strides=(2, 3),
            activation="relu",
        ),
        layers.MaxPooling2D(pool_size=(2, 3), strides=(2, 3)),
        layers.Flatten(),
        layers.Dense(64, activation="relu"),
        layers.Dropout(0.2),
        layers.Dense(3, activation="softmax"),
    ],
    name="yz_projection_cnn",
)
cnn_model.compile(
    optimizer=keras.optimizers.Adam(),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)
cnn_model.summary()


In [ ]:
cnn_history = cnn_model.fit(
    cnn_train_dataset,
    validation_data=cnn_validation_dataset,
    epochs=MAX_EPOCHS,
    callbacks=[early_stopping()],
    verbose=1,
)


In [ ]:
plot_history(cnn_history)

cnn_probability = cnn_model.predict(cnn_test_dataset, verbose=0)
cnn_prediction = cnn_probability.argmax(axis=1)
y_cnn_test = all_labels[cnn_test_indices]
show_evaluation(y_cnn_test, cnn_prediction, CLASS_NAMES)


The dense network receives compact, physics-motivated track variables; the CNN receives only the $YZ$ projection. Their scores therefore measure different representations as well as different architectures. The CNN also uses a smaller teaching sample by default, so the two test results should not be treated as a controlled architecture comparison.


## Exercises

1. Set `CNN_EVENTS_PER_CLASS = None` and measure the change in training time and test performance.
2. Add the $XZ$ and $XY$ projections as extra image channels. Keep the same event split for a fair comparison.
3. Change the dense-network width, depth, dropout rate, or learning rate. Make every choice using the validation sample, then evaluate the selected configuration once on the test sample.
4. Compare class weighting with balanced sampling for the CNN.
5. Inspect confidently misclassified events and look for recurring track topologies.

Further reading: [TensorFlow classification tutorial](https://www.tensorflow.org/tutorials/keras/classification), [TensorFlow `tf.data` guide](https://www.tensorflow.org/guide/data), and the [HEP ML Living Review](https://iml-wg.github.io/HEPML-LivingReview/).
